# Bullinger whole-page recognition: upload and inspect

**Upload a page → contour line segmentation → public Bullinger TrOCR → mBERT with historical aliases → modern place candidates.**

1. In Colab, select **Runtime → Change runtime type → T4 GPU**. CPU inference is also supported but slower.
2. Run the cells from top to bottom. In step 2, upload one JPG, PNG, WebP, or single-page TIFF image.
3. Inspect the segmentation preview, line images and transcriptions, and place candidates. Download the JSON or the complete results ZIP in step 6.

The first run downloads the environment and model weights and may take several minutes. No Drive mount or Hugging Face login is required.
To try another image in the same session, start again at step 2. Each run uses a new output directory and preserves earlier results.

This notebook uses the verified [GitHub pipeline revision fb7a912](https://github.com/sssstong55t-sudo/Bullinger_htr/tree/fb7a912160c0ab51fae8bf20efb23287e65b96e8).
It targets German and Latin Bullinger manuscripts. Review predictions on new images manually. CER/WER cannot be computed without explicit reference transcriptions.
All modern place candidates are retained. Even a single candidate requires review. Missing names or coordinates remain null.


## 1. Install dependencies and download models (once per session)
Use an isolated Python environment to avoid changing libraries already loaded by Colab. HTR uses the GPU when available; mBERT runs on CPU.


In [ ]:
import os, sys, json, subprocess, hashlib, shutil, zipfile
from pathlib import Path
from urllib.request import urlopen, Request

COMMIT = 'fb7a912160c0ab51fae8bf20efb23287e65b96e8'
BASE = Path('/content/bullinger_upload_demo')
BASE.mkdir(parents=True, exist_ok=True)
REPO = BASE / ('repo_' + COMMIT[:7])
PY = BASE / 'venv/bin/python'

def run(args, cwd=None, env=None):
    command = [str(x) for x in args]
    with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if not REPO.exists():
    run(['git', 'clone', 'https://github.com/sssstong55t-sudo/Bullinger_htr.git', REPO])
run(['git', '-C', REPO, 'checkout', '--detach', COMMIT])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == COMMIT

if not PY.exists():
    run([sys.executable, '-m', 'pip', 'install', '--quiet', 'virtualenv'])
    run([sys.executable, '-m', 'virtualenv', BASE / 'venv'])
installed = BASE / 'environment_v2.ready'
if not installed.exists():
    run([PY, '-m', 'pip', 'install', '--upgrade', 'pip'])
    # CUDA 12.4 wheel also runs on CPU. No optional GLiNER download is needed.
    run([PY, '-m', 'pip', 'install', 'torch==2.6.0',
         '--index-url', 'https://download.pytorch.org/whl/cu124'])
    run([PY, '-m', 'pip', 'install', 'transformers==4.51.0', 'accelerate==1.15.0',
         'huggingface-hub==0.36.2', 'Pillow>=10,<13', 'numpy>=1.24,<3',
         'scipy>=1.11,<2', 'sentencepiece==0.2.2', 'protobuf>=4,<7'])
    installed.touch()
run([PY, '-c', "import torch; from transformers import ViTImageProcessor, VisionEncoderDecoderModel, AutoModelForTokenClassification; print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slower transcription)')"])

os.environ['HF_HUB_DISABLE_XET'] = '1'
run([PY, REPO / 'scripts/download_public_trocr.py'], cwd=REPO)
archive = REPO / 'models/bullinger_mbert_20261022.zip'
expected = '92cd0da3f00234a9045831ed5ca70c6c7b43d8b310fc681b1df104ed12a88683'
url = 'https://github.com/sssstong55t-sudo/Bullinger_htr/releases/download/v1.0.0-mbert-20261022/bullinger_mbert_20261022.zip'
if not archive.exists() or sha256(archive) != expected:
    temp = archive.with_suffix('.part')
    print('Downloading mBERT weights...')
    with urlopen(Request(url, headers={'User-Agent': 'Bullinger-Colab'}), timeout=120) as response, temp.open('wb') as out:
        shutil.copyfileobj(response, out, length=8 * 1024 * 1024)
    if sha256(temp) != expected:
        temp.unlink()
        raise RuntimeError('mBERT checksum verification failed. Run this cell again.')
    temp.replace(archive)
with zipfile.ZipFile(archive) as z:
    destination = (REPO / 'models').resolve()
    for entry in z.infolist():
        if not (destination / entry.filename).resolve().is_relative_to(destination):
            raise ValueError('The model archive contains an invalid path.')
    z.extractall(destination)
assert sha256(REPO / 'models/bullinger_mbert_20261022/model.safetensors') == '055be2e5c923a4e6a843e120f73cffb66eaa3cdf65a686faf6d776dfeb99ae20'
INFERENCE_ENV = {**os.environ, 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'PYTHONUNBUFFERED': '1'}
print('Environment and models are ready. Run step 2 to upload an image.')


## 2. Upload one page image
The uploaded image is displayed below. Check its orientation and legibility. Export PDFs as images and split multipage TIFFs before uploading.


In [ ]:
from google.colab import files
from IPython.display import display, HTML
from PIL import Image, ImageOps
import io, uuid
from datetime import datetime, timezone

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one page image, then run this cell again.')
original_name, image_bytes = next(iter(uploaded.items()))
with Image.open(io.BytesIO(image_bytes)) as source:
    if getattr(source, 'n_frames', 1) != 1:
        raise ValueError('Split multipage or animated images into individual pages first.')
    oriented = ImageOps.exif_transpose(source).convert('RGBA')
    page = Image.alpha_composite(Image.new('RGBA', oriented.size, 'white'), oriented).convert('RGB')
RUN = BASE / 'runs' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
(RUN / 'input').mkdir(parents=True)
PAGE = RUN / 'input/page.png'
page.save(PAGE)
upload_info = {'original_filename': original_name,
               'original_sha256': hashlib.sha256(image_bytes).hexdigest()}
(RUN / 'upload.json').write_text(json.dumps(upload_info, ensure_ascii=False, indent=2), encoding='utf-8')
preview = page.copy()
preview.thumbnail((850, 1100))
print(f'Image: {original_name}; size: {page.width} x {page.height}')
display(preview)


## 3. Segment the page and inspect the preview
Red outlines mark line contours. Blue numbers correspond to the transcriptions below. Marginal notes, multiple columns, or touching lines may cause segmentation errors that affect recognition.


In [ ]:
# Re-running this step creates fresh output paths for all subsequent steps.
ATTEMPT = RUN / ('attempt_' + uuid.uuid4().hex[:8])
ATTEMPT.mkdir()
CROPS = ATTEMPT / 'crops'
run([PY, REPO / 'scripts/page_segmentation_loose.py', '--image', PAGE,
     '--output-dir', CROPS], cwd=REPO, env=INFERENCE_ENV)
manifest = json.loads((CROPS / 'manifest.json').read_text())
print(f"Detected {len(manifest['lines'])} lines.")
with Image.open(CROPS / 'preview.png') as im:
    preview = im.copy()
preview.thumbnail((1100, 1500))
display(preview)
if not manifest['lines']:
    raise ValueError('No text lines were found. Try a legible image with the correct orientation.')


## 4. Transcribe lines and extract place mentions
Progress is printed for each line. The public TrOCR model uses eight-beam decoding. mBERT and the historical lexicons then process the actual recognized text.
Re-running this cell creates new prediction files.


In [ ]:
RESULT = ATTEMPT / ('recognition_' + uuid.uuid4().hex[:8])
RESULT.mkdir()
HTR = RESULT / 'htr.jsonl'
PLACES = RESULT / 'places.jsonl'
crop_paths = [CROPS / line['image'] for line in manifest['lines']]
run([PY, REPO / 'scripts/transcribe_images.py', '--images', *crop_paths,
     '--output', HTR, '--device', 'auto'], cwd=REPO, env=INFERENCE_ENV)
run([PY, REPO / 'scripts/extract_places.py', '--input', HTR,
     '--output', PLACES], cwd=REPO, env=INFERENCE_ENV)
print('Transcription and place extraction are complete. Run step 5 to inspect the results.')


## 5. Inspect results and generate JSON for your web application
Place mentions are highlighted in yellow. The candidate table includes names, official Bullinger place IDs, coordinates, and sources.
"One candidate: review required" means the lexicons returned one candidate. "Multiple candidates" means the mention remains ambiguous. "Unmatched" means no lexicon candidate was found.
Candidate coordinates do not establish the identity of a historical place mention.


In [ ]:
from collections import Counter
import base64, html

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def index_by_image(rows):
    indexed = {Path(row['image']).resolve(): row for row in rows}
    assert len(indexed) == len(rows), 'Duplicate line images were found.'
    return indexed

htr = index_by_image(read_jsonl(HTR))
ner = index_by_image(read_jsonl(PLACES))
assert set(htr) == set(ner) == set(path.resolve() for path in crop_paths), 'Line images do not align across pipeline stages.'
counts = Counter()
lines = []
for segment in manifest['lines']:
    crop = (CROPS / segment['image']).resolve()
    prediction, entities = htr[crop], ner[crop]
    assert prediction['text'] == entities['text']
    assert prediction['image_sha256'] == sha256(crop)
    for mention in entities['places']:
        start, end = mention['start'], mention['end']
        assert 0 <= start < end <= len(prediction['text'])
        assert prediction['text'][start:end] == mention['text']
        candidates = mention['official_place_candidates']
        assert len({c['official_place_id'] for c in candidates}) == len(candidates)
        status = 'unmatched' if not candidates else ('candidate_for_review' if len(candidates) == 1 else 'ambiguous')
        assert mention['place_link_status'] == status
        counts[status] += 1
    lines.append({'line_id': segment['line_id'], 'image': 'crops/' + segment['image'],
                  'bbox': segment['bbox'], 'polygon': segment['polygon'],
                  'text': prediction['text'], 'places': entities['places']})
payload = {
    'page': {'page_id': RUN.name, **upload_info, 'image': 'input/page.png',
             'image_sha256': sha256(PAGE), 'width': manifest['page_size'][0],
             'height': manifest['page_size'][1], 'coordinate_system': manifest['coordinate_system'],
             'reading_order': manifest['reading_order']},
    'pipeline': {'github_commit': COMMIT, 'segmentation': 'contours',
                 'htr_model': 'pstroe/bullinger-general-model',
                 'htr_revision': 'be32c883a4deb812ab6f461ae79f583c3f9edc53',
                 'place_extraction_mode': 'mbert_with_aliases',
                 'place_resolution': 'official lexicon candidates for review',
                 'text_offset_unit': 'Unicode code points; end exclusive'},
    'summary': {'line_count': len(lines), 'place_mention_count': sum(counts.values()),
                'place_link_status_counts': {key: counts[key] for key in
                    ['candidate_for_review', 'ambiguous', 'unmatched']},
                'mentions_with_candidate_coordinates': sum(
                    any(c['latitude'] is not None and c['longitude'] is not None
                        for c in m['official_place_candidates']) for line in lines for m in line['places'])},
    'lines': lines}
JSON_PATH = RESULT / 'page_response.json'
JSON_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(RESULT / 'transcription.txt').write_text('\n'.join(line['text'] for line in lines) + '\n', encoding='utf-8')

def esc(value):
    return html.escape('—' if value is None else str(value))

def marked_text(text, mentions):
    # Union overlapping spans so the display preserves every original character.
    spans = []
    for mention in sorted(mentions, key=lambda m: (m['start'], m['end'])):
        start, end = mention['start'], mention['end']
        if spans and start <= spans[-1][1]:
            spans[-1][1] = max(spans[-1][1], end)
        else:
            spans.append([start, end])
    chunks, cursor = [], 0
    for start, end in spans:
        chunks += [esc(text[cursor:start]), '<mark>' + esc(text[start:end]) + '</mark>']
        cursor = end
    return ''.join(chunks) + esc(text[cursor:])

labels = {'candidate_for_review': 'One candidate: review required', 'ambiguous': 'Multiple candidates', 'unmatched': 'Unmatched'}
body = [f"<h1>Bullinger page recognition results</h1><p>{esc(original_name)} · {len(lines)} lines · {sum(counts.values())} place mentions</p>",
        '<p>Place mentions are highlighted in yellow. Modern place names are lexicon candidates that require review.</p>']
for line in lines:
    with Image.open(CROPS / Path(line['image']).name) as im:
        thumb = im.copy()
    thumb.thumbnail((1000, 240))
    buffer = io.BytesIO()
    thumb.save(buffer, format='PNG')
    data = base64.b64encode(buffer.getvalue()).decode('ascii')
    body.append(f"<section><h3>{esc(line['line_id'])}</h3><div class='pair'><img src='data:image/png;base64,{data}'><p>{marked_text(line['text'], line['places'])}</p></div></section>")
body.append('<h2>Modern place candidates</h2>')
if not sum(counts.values()):
    body.append('<p>No place mentions were detected. Each line has an empty places array in the JSON.</p>')
else:
    body.append('<div class="scroll"><table><tr><th>Line</th><th>Mention</th><th>Status</th><th>Candidate name</th><th>Place ID</th><th>Latitude</th><th>Longitude</th><th>Source</th></tr>')
    for line in lines:
        for mention in line['places']:
            for candidate in mention['official_place_candidates'] or [{}]:
                values = [line['line_id'], mention['text'], labels[mention['place_link_status']],
                          candidate.get('canonical_name'), candidate.get('official_place_id'),
                          candidate.get('latitude'), candidate.get('longitude'), candidate.get('source')]
                body.append('<tr>' + ''.join('<td>' + esc(v) + '</td>' for v in values) + '</tr>')
    body.append('</table></div>')
style = 'body{font-family:system-ui,sans-serif;margin:24px;color:#222}section{border-top:1px solid #ddd;padding:12px 0}.pair{display:flex;gap:20px;align-items:center;flex-wrap:wrap}.pair img{max-width:100%;width:500px}.pair p{flex:1;min-width:240px;white-space:pre-wrap;font-size:18px}mark{background:#ffe58a}table{border-collapse:collapse}td,th{border:1px solid #ddd;padding:8px;text-align:left}.scroll{overflow:auto}'
report = '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Bullinger recognition results</title><style>' + style + '</style></head><body>' + ''.join(body) + '</body></html>'
(RESULT / 'report.html').write_text(report, encoding='utf-8')
display(HTML('<style>' + style + '</style>' + ''.join(body)))
print('JSON created:', JSON_PATH.name)


## 6. Download results
The JSON can be consumed by your web application. `start/end` are Unicode code-point offsets into the exact transcription, with an exclusive end.
In JavaScript, use `Array.from(line.text).slice(start, end).join('')` to retrieve a mention.
Image paths in the JSON are relative to the ZIP root. The ZIP includes the page image, line crops, segmentation preview, transcription, raw JSONL, and an offline `report.html`.
Download the results you want to keep before the Colab session ends.


In [ ]:
# Run this cell to download the JSON for your web application.
files.download(str(JSON_PATH))


In [ ]:
# Optional: download all results. Extract the ZIP and open report.html in a browser.
ZIP_PATH = RESULT / 'bullinger_page_results.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PAGE, 'input/page.png')
    for path in sorted(CROPS.iterdir()):
        if path.is_file():
            z.write(path, 'crops/' + path.name)
    for name in ['page_response.json', 'transcription.txt', 'report.html', 'htr.jsonl', 'places.jsonl']:
        z.write(RESULT / name, name)
files.download(str(ZIP_PATH))
